# K=64 canonical runs, two cards: pairlatw64lab60fp64, seeds 0, 1

so3c_message_set with beams + channels 8 + --keep-best --pair-lab --pair-latent 8 --act-hidden 64 --msg-dim 24 --scalar-dim 24 --pair-lab-fp64, 60 epochs, one seed per T4 in
parallel. T4 with torch 2.11.0+cu128 pinned and asserted. References: the
60-epoch headline (0.98363, 1140.8, 0.16276) (torch 2.10, seed 0) and its 120-epoch continuation (0.98392, 1201.9, 0.16184).

In [ ]:
import subprocess, time
T0 = time.time()
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True,
                     text=True).stdout.strip())

In [ ]:
import base64, io, tarfile, pathlib
CODE_B64 = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE_B64)), mode="r:gz").extractall("/kaggle/working/repo")
print(sorted(p.name for p in pathlib.Path("/kaggle/working/repo").iterdir()))

In [ ]:
import subprocess, sys

def run(args, cwd="/kaggle/working/repo"):
    """Run a repo module, stream nothing, return (ok, tail-of-output)."""
    r = subprocess.run([sys.executable, "-m"] + args, cwd=cwd,
                       capture_output=True, text=True)
    out = r.stdout[-4000:]
    if r.returncode != 0:
        out += "\n--- stderr ---\n" + r.stderr[-3000:]
    print(out)
    return r.returncode == 0

In [ ]:
import subprocess, sys, time
WANT = "2.11.0+cu128"
name = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
print("device:", name)
assert "T4" in name, "paired with T4 runs only; got %r -- stopping before any GPU time is spent" % name

def torch_version():
    return subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"],
                          capture_output=True, text=True).stdout.strip()

v = torch_version()
if v != WANT:
    print("torch", v, "- installing", WANT)
    t0 = time.perf_counter()
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.11.0",
                        "--index-url", "https://download.pytorch.org/whl/cu128"],
                       capture_output=True, text=True)
    print("pip rc", r.returncode, "in %.0f s" % (time.perf_counter() - t0))
    if r.returncode:
        print(r.stderr[-2000:])
        raise SystemExit("could not install torch " + WANT)
    v = torch_version()
assert v == WANT, "torch %s, expected %s" % (v, WANT)
print("torch", v, "on", name)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "torchdiffeq"], check=True)

In [ ]:
import os, subprocess, sys

def launch(args, gpu, log):
    """Start a repo module on one card; its output goes to `log`."""
    # unbuffered, with faulthandler: a killed or crashed child still leaves
    # its last lines and a traceback in `log`
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1",
               PYTHONFAULTHANDLER="1")
    return subprocess.Popen([sys.executable, "-m"] + args, cwd="/kaggle/working/repo",
                            env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT)

def tail(log, n=3000):
    return open(log).read()[-n:]

cards = [l for l in subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                   capture_output=True, text=True).stdout.splitlines() if l.strip()]
NGPU = len(cards)
print("cards:", cards)

In [ ]:
import glob, pathlib
cands = glob.glob("/kaggle/input/**/top_tagging_train.npz", recursive=True)
assert cands, "attach the K=128 data-prep kernel output"
DATA = str(pathlib.Path(cands[0]).parent)
TAG, SEEDS, EPOCHS, MAX_SECONDS = 'pairlatw64lab60fp64', [0, 1], 60, 39000
OUT = "/kaggle/working/results_k64/" + TAG
CKPT = "/kaggle/working/checkpoints/" + TAG
LOGS = "/kaggle/working/logs/" + TAG
BASE = ['--cache-dir', 'DATA', '--representation', 'constituents', '--canonical-splits', '--normalize', 'global', '--device', 'cuda', '--dtype', 'float32', '--batch-size', '256', '--n-constituents', '64', '--eval-chunk-size', '1024', '--models', 'so3c_message_set', '--rounds', '3', '--beams', '--channels', '8', '--compile', '--keep-best', '--pair-lab', '--pair-latent', '8', '--act-hidden', '64', '--msg-dim', '24', '--scalar-dim', '24', '--pair-lab-fp64']
BASE[BASE.index("DATA")] = DATA
for d in (OUT, LOGS):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
pathlib.Path(OUT, "bundle_sha256.txt").write_text('41767898acc6' + chr(10))
print("data:", DATA, "| bundle", '41767898acc6')

In [ ]:
# On-card guard: the forward pass must be the published function on this
# GPU too, in float64. "1 passed" is required; a skip means no CUDA.
import subprocess, sys
g = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider",
                    "tests/test_so3c_models.py", "-k", "golden_cuda"],
                   cwd="/kaggle/working/repo", capture_output=True, text=True)
print(g.stdout[-1500:], g.stderr[-1500:])
assert "1 passed" in g.stdout, "golden guard did not pass on the card"

In [ ]:
# Smoke on both cards at once, as the real runs will start, each card with
# its own seed, so both seeds' initial weights go through the first-batch
# compile check (on 20k jets' normalisation, not the full set's): one epoch
# on 20k jets per card. Each must write a result.
import glob, json, time
smoke = []
for i in range(min(NGPU, len(SEEDS))):
    d = "/kaggle/working/smoke/%s/card%d" % (TAG, i)
    smoke.append((d, launch(["benchmarks.run_top_tagging"] + BASE +
                            ["--epochs", "1", "--max-train-samples", "20000",
                             "--seed", str(SEEDS[i]), "--results-dir", d],
                            i, LOGS + "/smoke_card%d.log" % i)))
codes = [p.wait() for _, p in smoke]
works = []
for i, ((d, _), rc) in enumerate(zip(smoke, codes)):
    found = glob.glob(d + "/*.json")
    print("card %d: exit %d, result %s" % (i, rc, bool(found)))
    if found:
        r = json.load(open(found[0]))
        print("   seed %d: %.0f s, AUC %.4f, compile %s, %s"
              % (r["seed"], r["walltime_sec"], r["test_metrics"]["test_auc"],
                 r.get("compile_check"), r["env"]))
        works.append(i)
    else:
        print(tail(LOGS + "/smoke_card%d.log" % i))
assert works, "the smoke run failed on every card"
PARALLEL = len(works) >= len(SEEDS)
print("parallel:" if PARALLEL else "only card(s) %s work: seed %d alone" % (works, SEEDS[0]),
      "seeds", SEEDS if PARALLEL else SEEDS[:1])

In [ ]:
import glob, time
todo = SEEDS if PARALLEL else SEEDS[:1]
procs = []
for i, s in enumerate(todo):
    gpu = i if PARALLEL else works[0]
    procs.append((s, launch(["benchmarks.run_top_tagging"] + BASE +
                            ["--epochs", str(EPOCHS), "--seed", str(s),
                             "--results-dir", OUT + "/seed%d" % s,
                             "--ckpt-dir", CKPT + "/seed%d" % s,
                             "--resume", "--max-seconds", str(MAX_SECONDS)],
                            gpu, LOGS + "/seed%d.log" % s)))
    print("seed %d on card %d" % (s, gpu), flush=True)
BAD = []
for s, p in procs:
    rc = p.wait()
    print("=== seed %d: exit %d ===" % (s, rc))
    print(tail(LOGS + "/seed%d.log" % s, 2000))
    if rc != 0 or not glob.glob(OUT + "/seed%d/*.json" % s):
        BAD.append(s)
# a failed seed is reported after the summary below, so the other seed's
# result is still printed
print("failed seeds:", BAD or "none")
print("kernel time %.2f h" % ((time.time() - T0) / 3600))

In [ ]:
# Every result against the K=64 headline: the 60-epoch run (torch 2.10, seed 0)
# and its continuation to 120 epochs. Same-seed runs in different kernels do
# not reproduce on this stack, so these are unpaired comparisons.
import json, glob
REF60, REF120 = (0.98363, 1140.8, 0.16276), (0.98392, 1201.9, 0.16184)
for f in sorted(glob.glob(OUT + "/seed*/*.json")):
    r = json.load(open(f)); t = r["test_metrics"]; env = r["env"]; bo = r["batch_order"]
    h = r["history"]; vl = [e["val_loss"] for e in h]
    print("seed %d: %d epochs, %.2f h (%.0f s/epoch) | torch %s on %s | batch order epoch 1 %s"
          % (r["seed"], r["epochs_run"], r["walltime_sec"] / 3600,
             r["walltime_sec"] / max(r["epochs_run"], 1), env["torch"], env.get("gpu"), bo["epoch1"]))
    assert env["torch"] == "2.11.0+cu128" and "T4" in env.get("gpu", ""), env
    print("  AUC %.5f  rej@0.3 %.1f  rej@0.5 %.1f  final val loss %.5f  (lowest %.5f at epoch %d)"
          % (t["test_auc"], t["bg_rej_30"], t.get("bg_rej_50", float("nan")), vl[-1],
             min(vl), h[vl.index(min(vl))]["epoch"]))
    if r["epochs_run"] < EPOCHS:
        print("  INCOMPLETE (epoch %d of %d): stopped on the time cap; finish before comparing"
              % (r["epochs_run"], EPOCHS))
        continue
    for name, ref in (("60-epoch headline", REF60), ("120-epoch headline", REF120)):
        print("  against the %s: dAUC %+.5f, d rej@0.3 %+.1f, d final val loss %+.5f"
              % (name, t["test_auc"] - ref[0], t["bg_rej_30"] - ref[1], vl[-1] - ref[2]))
    if r.get("test_metrics_best_val"):
        tb = r["test_metrics_best_val"]
        print("  best-val weights (epoch %d): AUC %.5f, rej@0.3 %.1f"
              % (r["best_val_epoch"], tb["test_auc"], tb["bg_rej_30"]))
assert not BAD, "seeds failed (exit code or no result): %s" % BAD